In [1]:
from datetime import datetime

import gymnasium as gym
import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter

from src.agents.cart_ppo import PPOCartAgent
from src.utils import save_checkpoint


In [2]:
LR = 2.5e-4
num_episodes = 6000
K_EPOCH = 4
REPEAT = 4

In [3]:
run_name = f"ppo_lr{LR}_ne{num_episodes}_k{K_EPOCH}_r{REPEAT}_{datetime.now():%Y%m%d_%H%M%S}"
writer = SummaryWriter(f"./logs/{run_name}")

In [4]:
env = gym.make("CartPole-v1", render_mode=None)
agent = PPOCartAgent(env=env, learning_rate=LR)

# Regular training loop

In [5]:
%%script false --no-raise-error

episode_durations = []
for e in range(num_episodes):
    state, _ = env.reset()

    done = False
    states = []


    # FIRST PASS:
    actions = []
    probs = []
    rewards = []
    running_batch = []
    t = 0
    while not done:
        states.append(state)

        action, prob = agent.get_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        probs.append(prob)
        actions.append(action)
        rewards.append(reward)

        if agent.update_critic(running_batch):
            running_batch = []

        running_batch.append((state, action, reward, next_state, done))
        state = next_state
        t += 1



    # TRAINING PASS:
    old_log_probs = torch.stack(probs).detach()
    states_t = torch.tensor(np.array(states), dtype=torch.float32) 
    actions_t = torch.tensor(np.array(actions), dtype=torch.int8) 
        
    for _ in range(K_EPOCH):
        with torch.no_grad():
            critiques = agent.critic(states_t).squeeze(-1)
        new_log_probs = agent.evaluate_action(states_t, actions_t)
        ratio = torch.exp(new_log_probs - old_log_probs)
        agent.update(rewards, ratio, critiques)

    writer.add_scalar("duration/frame_skip", t, e+1)

print('Complete')

# Frame skip training loop

In [6]:
episode_durations = []
for e in range(num_episodes):
    state, _ = env.reset()

    done = False
    states = []


    # FIRST PASS:
    actions = []
    probs = []
    rewards = []
    running_batch = []
    t = 0
    while not done:
        states.append(state)

        action, prob = agent.get_action(state)

        total_reward = 0
        for _ in range(REPEAT):
            next_state, reward, terminated, truncated, _ = env.step(action)
            t += 1
            
            total_reward += reward

            done = terminated or truncated
            if done:
                break
        
        probs.append(prob)
        actions.append(action)
        rewards.append(total_reward)

        if agent.update_critic(running_batch):
            running_batch = []

        running_batch.append((state, action, reward, next_state, done))
        state = next_state



    # TRAINING PASS:
    old_log_probs = torch.stack(probs).detach()
    states_t = torch.tensor(np.array(states), dtype=torch.float32) 
    actions_t = torch.tensor(np.array(actions), dtype=torch.int8) 
        
    for _ in range(K_EPOCH):
        with torch.no_grad():
            critiques = agent.critic(states_t).squeeze(-1)
        new_log_probs = agent.evaluate_action(states_t, actions_t)
        ratio = torch.exp(new_log_probs - old_log_probs)
        agent.update(rewards, ratio, critiques)

    writer.add_scalar("duration", t, e+1)

print('Complete')

Complete


In [7]:
test_env = gym.make("CartPole-v1", render_mode=None)
test_agent = PPOCartAgent(env, 0, 0)

test_agent.policy.load_state_dict(agent.policy.state_dict())

test_reward = []

for e in range(1000):
    state, _ = test_env.reset()
    done = False
    total_reward = 0
    while not done:
        action = test_agent.act(state)
        next_state, reward, terminated, truncated, _ = test_env.step(action)

        done = terminated or truncated
        state = next_state
        total_reward += reward

    writer.add_scalar("eval/reward", total_reward, e+1)
    test_reward.append(total_reward)

print("avg: ", np.average(test_reward))

avg:  500.0


In [10]:
writer.add_hparams(
    {"lr": LR, "num_episodes": num_episodes, "k_epoch": K_EPOCH},
    {"final_avg_reward": np.average(test_reward)},
    run_name=".", 
)

In [8]:
writer.close()

In [9]:
if np.average(test_reward) > 495:
    print("Saved")
    save_checkpoint(agent.policy, run_name)

Saved
